In [1]:
from nilearn import plotting
%matplotlib inline
from os.path import join as opj
import json
from nipype.interfaces.base import Bunch
from nipype.interfaces.spm import Level1Design, EstimateModel, EstimateContrast, SPMCommand, Info, model
from nipype.interfaces.matlab import MatlabCommand
from nipype.interfaces.freesurfer import FSCommand
from nipype.algorithms.modelgen import SpecifySPMModel, SpecifyModel
from nipype.interfaces.utility import Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
from nipype import Workflow, Node
from bids.layout import BIDSLayout
from glob import glob
from scipy import io, stats
from itertools import chain
import pandas as pd
import numpy as np
import pytest as pt
import nibabel as nb
import nipype

MatlabCommand.set_default_paths('~/spm12')
MatlabCommand.set_default_matlab_cmd("/usr/local/MATLAB/R2022b/bin/matlab")
#SPMCommand.set_mlab_paths(matlab_cmd="/usr/local/MATLAB/R2022b/bin/matlab")

fs_dir = '/mnt/d/data/ds-mlearn/derivatives/freesurfer'
FSCommand.set_default_subjects_dir(fs_dir)

In [2]:
import nipype.interfaces.spm as spm
#mlab = MatlabCommand()
#mlab.inputs.script = "spm ver"
#res = mlab.run()

In [54]:
nipype.__file__

'/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/__init__.py'

In [3]:
experiment_dir = "/SPM"
output_dir = "nipype"
working_dir = "workingdir"
layout = BIDSLayout("/mnt/d/data/ds-mlearn/", derivatives=True)

# list of subject identifiers
subject_list = layout.get_subjects()

# TR of functional images
with open(
    "/mnt/d/data/ds-mlearn/derivatives/fmriprep/sub-01/func/sub-01_task-learn_run-1_space-T1w_desc-preproc_bold.json",
    "rt",
) as fp:
    task_info = json.load(fp)
TR = task_info["RepetitionTime"]
Nslices = 40
refSlice = round(Nslices / 2)

# Smoothing widths used during preprocessing
fwhm = 6

In [4]:
def get_subject_info(subject):
    from glob import glob
    import numpy as np
    import pandas as pd
    from scipy import io, stats
    from nipype.interfaces.base import Bunch

    subject_info = []
    rpe_path = (
        f"/mnt/d/multlearn-sns/Modelling/Fitting/bestFittingVals/sub-{subject}/rpe*.mat"
    )
    fn = glob(rpe_path)

    assert len(fn) == 1
    fn = fn[0]

    rpe_data = np.nan_to_num(stats.zscore(io.loadmat(fn)["rpe"],nan_policy='omit',axis=1))

    rpe = (
        pd.DataFrame(
            rpe_data,
            index=pd.Index(np.arange(1, 6 + 1), name="run"),
            columns=pd.Index(np.arange(1, 60 + 1), name="trial_nr"),
        )
        .stack()
        .to_frame("rpe")
    )
    
    surprise_path = (
        f"/mnt/d/multlearn-sns/Modelling/Fitting/bestFittingVals/sub-{subject}/spe*.mat"
    )
    fn2 = glob(surprise_path)
    assert len(fn2) == 1
    fn2 = fn2[0]
    surprise_data = np.nan_to_num(stats.zscore(io.loadmat(fn2)["spe"],nan_policy='omit',axis=1))
    surprise = (
        pd.DataFrame(
            surprise_data,
            index=pd.Index(np.arange(1, 6 + 1), name="run"),
            columns=pd.Index(np.arange(1, 60 + 1), name="trial_nr"),
        )
        .stack()
        .to_frame("spe")
    )
    
    functional_runs = []
    
    for run in range(1, 7):
        onsets = []
        durations = []
        conditions = []

        events_file = pd.read_csv(
            f"/mnt/d/data/ds-mlearn/derivatives/fmriprep/sub-{subject}/func/sub-{subject}_task-learn_run-{run}_events.tsv",
            delimiter="\t",
        )
        events_file_sorted = events_file.sort_values(by=["onset"])
        events_file_sorted["trial_nr"] = events_file_sorted["trial_nr"].ffill()
        events_file_sorted["runType"] = events_file_sorted["runType"].ffill()
        run_type = events_file_sorted["runType"][0]

        confounds = pd.read_csv(
            f"/mnt/d/data/ds-mlearn/derivatives/fmriprep/sub-{subject}/func/sub-{subject}_task-learn_run-{run}_desc-confounds_timeseries.tsv",
            delimiter="\t",
        )

        confounds = confounds.loc[
            :,
            [
                "trans_x",
                "trans_y",
                "trans_z",
                "rot_x",
                "rot_y",
                "rot_z",
                "a_comp_cor_00",
                "a_comp_cor_01",
                "a_comp_cor_02",
                "a_comp_cor_03",
                "a_comp_cor_04",
            ],
        ]

        physio_path = f"/mnt/d/data/ds-mlearn/derivatives/fmriprep/sub-{subject}/beh/physio/RegPhysio_sub-{subject}_run_{run}.mat"
        fn3 = glob(physio_path)
        assert len(fn3) == 1
        fn3 = fn3[0]

        physio = io.loadmat(fn3, simplify_cells=True)["physio"]["model"]
        physio = pd.DataFrame(
            data=physio["R"],
            columns=physio["R_column_names"],
        )
     
        regressors = pd.concat([confounds, physio], axis=1)
        regressor_names = regressors.columns.values.tolist()
        

        for group in events_file_sorted.groupby("trial_type"):
            conditions.append(str(group[0].capitalize() + run_type.capitalize()))
            onsets.append(group[1]["onset"].tolist())
            durations.append(group[1]["duration"].tolist())
        run_rpe = rpe.xs(run)
        run_surprise = surprise.xs(run)
        pmod = [
            Bunch(name=["surprise"], param=[run_surprise.values.tolist()], poly=[1]),
            Bunch(name=["rpe"], param=[run_rpe.values.tolist()], poly=[1]),
        ]

        subject_info.insert(
            run - 1,
            Bunch(
                conditions=conditions,
                onsets=onsets,
                durations=durations,
                pmod=pmod,
                tmod=None,
                orth=['No']*len(conditions),
                regressors=regressors.values.T.tolist(),
                regressor_names=regressor_names,
            ),
        )

    

        functional_run = glob(
            f"/mnt/d/data/ds-mlearn/derivatives/fmriprep/sub-{subject}/func/s6.sub-{subject}_task-learn_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii"
        )[0]
        functional_runs.append(functional_run)

    return subject_info, functional_runs

In [5]:
subject_info, functional_runs = get_subject_info(subject_list[0])

In [6]:
getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo",
)

In [7]:
getsubjectinfo.inputs.subject = subject_list[0]
result = getsubjectinfo.run()
print(result.outputs.subject_info[1])
print(result.outputs.functional_runs)

240116-19:54:23,699 nipype.workflow INFO:
	 [Node] Setting-up "getsubjectinfo" in "/tmp/tmpr_ngi7ut/getsubjectinfo".
240116-19:54:23,703 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240116-19:54:24,731 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 1.022454s.
Bunch(conditions=['ChoiceAudio', 'FeedbackAudio'], durations=[[1.84416436838092, 1.85939117723319, 1.74539755182559, 1.77655396205319, 2.49098733116898, 2.54304899584531, 2.5384736870592, 2.07103395587728, 2.44208375342805, 2.55523657851791, 2.24712651496975, 1.96105483412248, 1.94694342759158, 2.2209038350602, 2.44874921182691, 1.57981999380263, 2.1896709103903, 1.67420471581318, 1.8797446654753, 2.22470329835778, 1.51527716000328, 1.59186492327444, 1.6230729682693, 2.06916190250558, 2.07585133969951, 1.98674988429138, 1.72699494685457, 1.99650802459655, 2.14947461795236, 2.02719649004393, 1.99747762241168, 1.80624573683235, 2.44517480953164, 2.121

In [8]:
print(result.outputs.subject_info[5].orth)

['No', 'No']


In [43]:
def get_contrasts(subject_info):

    from nipype.interfaces.spm import EstimateContrast
    import os

    condition_names = ['ChoiceAudio', 'ChoiceTactile', 'FeedbackAudio', 'FeedbackTactile',
    'ChoiceAudioxsurprise^1', 'ChoiceTactilexsurprise^1', 'FeedbackAudioxrpe^1', 'FeedbackTactilexrpe^1']

    con01 = ['rpe', 'T', condition_names[6:], [1/6., 1/6.]]
    con02 = ['rpe_audio', 'T', [condition_names[6]], [1/3.]]
    con03 = ['rpe_tactile', 'T', [condition_names[7]], [1/3.]]
    con04 = ['rpe_audio < rpe_tactile', 'T', [condition_names[6], condition_names[7]], [-1/3., 1/3.]]
    con05 = ['rpe_tactile < rpe_audio', 'T', [condition_names[6], condition_names[7]], [1/3., -1/3.]]
    
    con06 = ['surprise', 'T', condition_names[4:6], [1/6., 1/6.]]
    con07 = ['surprise_audio', 'T', [condition_names[4]], [1/3.]]
    con08 = ['surprise_tactile', 'T', [condition_names[5]], [1/3.]]
    con09 = ['surprise_audio < surprise_tactile', 'T', [condition_names[4], condition_names[5]], [-1/3., 1/3.]]
    con10 = ['surprise_tactile < surprise_audio', 'T', [condition_names[4], condition_names[5]], [1/3., -1/3.]]

    con11 = ['rpe < surprise', 'T', condition_names[4:], [-1/6., 1/6., -1/6., 1/6., -1/6., 1/6.]]
    con12 = ['surprise < rpe', 'T', condition_names[4:], [1/6., -1/6., 1/6., -1/6., 1/6., -1/6.]]
    con13 = ['rpe_audio < surprise_audio', 'T', [condition_names[4], condition_names[6]], [1/3., -1/3.]]
    con14 = ['surprise_audio < rpe_audio', 'T', [condition_names[4], condition_names[6]], [-1/3., 1/3.]]
    con15 = ['rpe_tactile < surprise_tactile', 'T', [condition_names[5], condition_names[7]], [1/3., -1/3.]]
    con16 = ['surprise_tactile < rpe_tactile', 'T', [condition_names[5], condition_names[7]], [-1/3., 1/3.]]

    con17 = ['pmods', 'T', condition_names[4:8], [1/12., 1/12., 1/12., 1/12.]]
    con18 = ['pmods_audio', 'T', [condition_names[4], condition_names[6]], [1/6., 1/6.]]
    con19 = ['pmods_tactile', 'T', [condition_names[5], condition_names[7]], [1/6., 1/6.]]
    con20 = ['pmods_audio-pmods_tactile', 'T', condition_names[4:8], [-1/6., 1/6., -1/6., 1/6.]]
    con21 = ['pmods_tactile-pmods_audio', 'T', condition_names[4:8], [1/6., -1/6., 1/6., -1/6.]]

    con22 = ['feedback', 'T', condition_names[2:4], [1/6., 1/6.]]
    con23 = ['choice', 'T', condition_names[0:2], [1/6., 1/6.]]
    con24 = ['feedback < choice', 'T', condition_names[0:4], [1/6., 1/6., -1/6., -1/6.]]
    con25 = ['choice < feedback', 'T', condition_names[0:4], [-1/6., -1/6., 1/6., 1/6.]]

    con_list = [con01, con02, con03, con04, con05, con06, con07, con08, con09, con10, con11, con12, con13,
    con14, con15, con16, con17, con18, con19, con20, con21, con22, con23, con24, con25]


    return con_list


In [44]:
getcontrasts = Node(
    Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts,
    ),
    name="getcontrasts",
)

In [45]:
getcontrasts.inputs.subject_info = result.outputs.subject_info
result_con = getcontrasts.run()
print(result_con.outputs.contrasts)

240116-21:47:23,888 nipype.workflow INFO:
	 [Node] Setting-up "getcontrasts" in "/tmp/tmpxjdo0wqi/getcontrasts".
240116-21:47:24,15 nipype.workflow INFO:
	 [Node] Executing "getcontrasts" <nipype.interfaces.utility.wrappers.Function>
240116-21:47:24,23 nipype.workflow INFO:
	 [Node] Finished "getcontrasts", elapsed time 0.001721s.
[['rpe', 'T', ['FeedbackAudioxrpe^1', 'FeedbackTactilexrpe^1'], [0.16666666666666666, 0.16666666666666666]], ['rpe_audio', 'T', ['FeedbackAudioxrpe^1'], [0.3333333333333333]], ['rpe_tactile', 'T', ['FeedbackTactilexrpe^1'], [0.3333333333333333]], ['rpe_audio < rpe_tactile', 'T', ['FeedbackAudioxrpe^1', 'FeedbackTactilexrpe^1'], [-0.3333333333333333, 0.3333333333333333]], ['rpe_tactile < rpe_audio', 'T', ['FeedbackAudioxrpe^1', 'FeedbackTactilexrpe^1'], [0.3333333333333333, -0.3333333333333333]], ['surprise', 'T', ['ChoiceAudioxsurprise^1', 'ChoiceTactilexsurprise^1'], [0.16666666666666666, 0.16666666666666666]], ['surprise_audio', 'T', ['ChoiceAudioxsurprise^

In [46]:
modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs",
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec",
)

In [47]:
modelspec.inputs.subject_info = result.outputs.subject_info
modelspec.inputs.functional_runs = result.outputs.functional_runs

res_modelspec = modelspec.run()
print(res_modelspec.outputs.session_info)

240116-21:47:28,209 nipype.workflow INFO:
	 [Node] Setting-up "modelspec" in "/tmp/tmpc49r885u/modelspec".
240116-21:47:28,638 nipype.workflow INFO:
	 [Node] Executing "modelspec" <nipype.algorithms.modelgen.SpecifySPMModel>
240116-21:47:28,648 nipype.workflow INFO:
	 [Node] Finished "modelspec", elapsed time 0.005187s.
[{'cond': [{'name': 'ChoiceTactile', 'onset': [3.47770013161835, 11.1003976227662, 19.372291248174, 27.596148337946, 35.1763284688304, 42.2580048041546, 51.4684016129409, 59.4676873441217, 68.3234503465715, 77.2160862214819, 83.8103209850292, 91.6483885658777, 101.349533172408, 108.87321136494, 116.104077388172, 124.826389406196, 134.19941888961, 142.597129984186, 150.188198734525, 159.323195201642, 167.906970439996, 174.821857776725, 182.23581163173, 190.134337197493, 199.3318513603, 208.335738915708, 217.530857753145, 226.893352375404, 235.819457682047, 244.346177409956, 252.446293077589, 260.064744063167, 268.134572390468, 276.884240842166, 284.282854423881, 292.2905

In [48]:
res_modelspec.outputs.session_info[0]['cond'][1].keys()

dict_keys(['name', 'onset', 'duration', 'orth', 'pmod'])

In [49]:
# Level1Design - Generates an SPM design matrix
level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii",
        volterra_expansion_order=1,
    ),
    name="level1design",
)

In [50]:
level1design.inputs.session_info = res_modelspec.outputs.session_info
res_level1design = level1design.run()
print(res_level1design.outputs.spm_mat_file)

240116-21:47:33,768 nipype.workflow INFO:
	 [Node] Setting-up "level1design" in "/tmp/tmpaqlra1yp/level1design".
240116-21:47:34,336 nipype.workflow INFO:
	 [Node] Executing "level1design" <nipype.interfaces.spm.model.Level1Design>
240116-21:48:50,661 nipype.workflow INFO:
	 [Node] Finished "level1design", elapsed time 76.319281s.
/tmp/tmpaqlra1yp/level1design/SPM.mat


stty: 'standard input': Inappropriate ioctl for device


In [29]:
str(res_level1design.outputs.spm_mat_file)

'/tmp/tmpyxqppn8u/level1design/SPM.mat'

In [51]:
# EstimateModel - estimate the parameters of the model
level1estimate = Node(
    EstimateModel(estimation_method={"Classical": 1},write_residuals=False), name="level1estimate"
)

In [52]:
level1estimate.inputs.spm_mat_file = res_level1design.outputs.spm_mat_file
res_level1est = level1estimate.run()
print(res_level1est.outputs)

240116-21:48:50,776 nipype.workflow INFO:
	 [Node] Setting-up "level1estimate" in "/tmp/tmpvabqxjva/level1estimate".
240116-21:48:50,779 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: 'standard input': Inappropriate ioctl for device


240116-21:51:12,627 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 141.840895s.

ARcoef = <undefined>
Cbetas = <undefined>
RPVimage = /tmp/tmpvabqxjva/level1estimate/RPV.nii
SDbetas = <undefined>
SDerror = <undefined>
beta_images = ['/tmp/tmpvabqxjva/level1estimate/beta_0001.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0002.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0003.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0004.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0005.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0006.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0007.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0008.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0009.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0010.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0011.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0012.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0013.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0014.nii', '/tmp/tmpvabqxjva/level1estimate/beta_0015.nii', '/tmp/tm

In [53]:
# EstimateContrast - estimates contrasts
level1conest = Node(EstimateContrast(), name="level1conest")
level1conest.inputs.contrasts = result_con.outputs.contrasts
level1conest.inputs.spm_mat_file = res_level1est.outputs.spm_mat_file
level1conest.inputs.beta_images = res_level1est.outputs.beta_images
level1conest.inputs.residual_image = res_level1est.outputs.residual_image
res_level1con = level1conest.run()
print(res_level1con.outputs)

240116-21:51:12,645 nipype.workflow INFO:
	 [Node] Setting-up "level1conest" in "/tmp/tmpq2f65lgy/level1conest".
240116-21:51:12,734 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>


stty: 'standard input': Inappropriate ioctl for device


240116-21:51:29,101 nipype.workflow INFO:
	 [Node] Finished "level1conest", elapsed time 16.361916s.

con_images = ['/tmp/tmpq2f65lgy/level1conest/con_0001.nii', '/tmp/tmpq2f65lgy/level1conest/con_0002.nii', '/tmp/tmpq2f65lgy/level1conest/con_0003.nii', '/tmp/tmpq2f65lgy/level1conest/con_0004.nii', '/tmp/tmpq2f65lgy/level1conest/con_0005.nii', '/tmp/tmpq2f65lgy/level1conest/con_0006.nii', '/tmp/tmpq2f65lgy/level1conest/con_0007.nii', '/tmp/tmpq2f65lgy/level1conest/con_0008.nii', '/tmp/tmpq2f65lgy/level1conest/con_0009.nii', '/tmp/tmpq2f65lgy/level1conest/con_0010.nii', '/tmp/tmpq2f65lgy/level1conest/con_0011.nii', '/tmp/tmpq2f65lgy/level1conest/con_0012.nii', '/tmp/tmpq2f65lgy/level1conest/con_0013.nii', '/tmp/tmpq2f65lgy/level1conest/con_0014.nii', '/tmp/tmpq2f65lgy/level1conest/con_0015.nii', '/tmp/tmpq2f65lgy/level1conest/con_0016.nii', '/tmp/tmpq2f65lgy/level1conest/con_0017.nii', '/tmp/tmpq2f65lgy/level1conest/con_0018.nii', '/tmp/tmpq2f65lgy/level1conest/con_0019.nii', '/tmp/tmpq

In [52]:
from nipype.interfaces.base import (
    BaseInterface,
    BaseInterfaceInputSpec,
    traits,
    File,
    TraitedSpec,
    InputMultiPath,
)
import subprocess


class SnpmOneSampleTTestInputSpec(BaseInterfaceInputSpec):
    destination = traits.Directory(
        exists=True, desc="Output directory for SnPM results"
    )

    contrasts = traits.List(traits.Any, desc="List of contrast files")
    covariates = InputMultiPath(
        traits.Dict(key_trait=traits.Enum("c", "cname")),
        field="cov",
        desc="Covariate dictionary {c, cname}",
    )
    n_perms = traits.Int(
        #usedefault=True,
        #default_value=5000,
        field="nPerm",
        desc="Number of permutations",
    )
    var_smoothing = traits.List(
        #default_value=[0, 0, 0],
        #usedefault=True,
        field="vFWHM",
        desc="Smoothing kernel",
    )
    memory_usage = traits.Bool(
        #default_value=False,
        #usedefault=True,
        field="bVolm",
        desc="Memory usage (False = low, True = high)",
    )
    cluster_inference_none = traits.Int(
        0,
        xor=["cluster_inference_later", "cluster_inference_fast"],
        field="ST.ST_none",
        desc="No cluster inference 0",
    )
    cluster_inference_later = traits.Int(
        -1,
        xor=["cluster_inference_none", "cluster_inference_fast"],
        field="ST.ST_later",
        desc="Cluster inference slow -1",
    )
    cluster_inference_fast = traits.Float(
        xor=["cluster_inference_none", "cluster_inference_later"],
        field="ST.ST_U",
        desc="Cluster inference fast t-value",
    )
    masking_none = traits.Int(
        1,
        field="masking.tm.tm_none",
        xor=["thresh_mask_abs", "thresh_mask_rel"],
        desc="No masking 1",
    )
    thresh_mask_abs = traits.Float(
        field="masking.tm.tma.athresh",
        xor=["masking_none", "thresh_mask_rel"],
        desc="Absolute threshold masking (in voxels)",
    )
    thresh_mask_rel = traits.Float(
        field="masking.tm.tmr.rthresh",
        xor=["masking_none", "thresh_mask_abs"],
        desc="Relative threshold masking (proportion of global value)",
    )
    implicit_mask = traits.Enum(
        0,
        1,
        #usedefault=True,
        field="masking.im",
        desc="Implicit masking (0 = No, 1 = Yes)",
    )
    explicit_mask = traits.Any(
        traits.Any, field="masking.em", desc="Explicit mask files"
    )

    global_calc_omit = traits.Int(
        1,
        field="globalc.g_omit",
        xor=["global_calc_user", "global_calc_mean"],
        desc="Omit global calculation 1",
    )
    global_calc_user = traits.List(
        field="globalc.g_user",
        xor=["global_calc_omit", "global_calc_mean"],
        desc="User-defined global values",
    )
    global_calc_mean = traits.Int(
        1,
        field="globalc.g_mean",
        xor=["global_calc_omit", "global_calc_user"],
        desc="Mean global calculation 1",
    )
    no_grand_mean_scaling = traits.Int(
        1,
        field="globalm.gmsca.gmsca_no",
        xor=["grand_mean_scaling"],
        desc="No grand mean scaling 1",
    )
    grand_mean_scaling = traits.List(
        #default_value=[50],
        #usedefault=True,
        field="globalm.gmsca.gmsca_yes.gmscv",
        xor=["no_grand_mean_scaling"],
        desc="Grand mean scaling",
    )
    global_normalization = traits.Enum(
        1,
        2,
        3,
        mandatory=True,
        field="globalm.glonorm",
        desc="Global normalization (1 = None, 2 = Proportional, 3 = ANCOVA)",
    )


class SnpmOneSampleTTestOutputSpec(TraitedSpec):
    results = traits.List(traits.File, desc="List of SnPM result files")


class SnpmOneSampleTTest(BaseInterface):
    input_spec = SnpmOneSampleTTestInputSpec
    output_spec = SnpmOneSampleTTestOutputSpec
    _jobtype = "tools.snpm.des"
    _jobname = "OneSampT"

    def _run_interface(self, runtime):
        # Construct the SnPM command based on input specifications
        snpm_command = "function SnPM_script()\n"
        snpm_command += "addpath('~/spm12');\n"
        snpm_command += "spm_jobman('initcfg');\n"
        snpm_command += "matlabbatch{1}.spm.tools.snpm.des.OneSampT.DesignName = 'MultiSub: One Sample T test on diffs/contrasts';\n"
        snpm_command += "matlabbatch{1}.spm.tools.snpm.des.OneSampT.DesignFile = 'snpm_bch_ui_OneSampT';\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.dir = cellstr('{self.inputs.destination}');\n"
        snpm_command += "matlabbatch{1}.spm.tools.snpm.des.OneSampT.P = {...\n"

        # Add contrast files to the command
        for contrast in self.inputs.contrasts:
            snpm_command += f"'{contrast}';...\n"

        snpm_command += "};\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.nPerm = {self.inputs.n_perms};\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.vFWHM = {self.inputs.var_smoothing};\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.bVolm = {int(self.inputs.memory_usage)};\n"
        if self.inputs.cluster_inference_none:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.ST.ST_none = {self.inputs.cluster_inference_none};\n"
        elif self.inputs.cluster_inference_later:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.ST.ST_later = {self.inputs.cluster_inference_later};\n"
        elif self.inputs.cluster_inference_fast:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.ST.ST_U = {self.inputs.cluster_inference_fast};\n"
        if self.inputs.masking_none:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.masking.tm.tm_none = {self.inputs.masking_none};\n"
        elif self.inputs.thresh_mask_abs:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.masking.tm.tma.athresh = {self.inputs.thresh_mask_abs};\n"
        elif self.inputs.thresh_mask_rel:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.masking.tm.tmr.rthresh = {self.inputs.thresh_mask_rel};\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.masking.im = {self.inputs.implicit_mask};\n"
        if self.inputs.explicit_mask:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.masking.em = {self.inputs.explicit_mask};\n"
        else:
            snpm_command += "matlabbatch{1}.spm.tools.snpm.des.OneSampT.masking.em = cellstr('');\n"
        if self.inputs.global_calc_omit:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalc.g_omit = {self.inputs.global_calc_omit};\n"
        elif self.inputs.global_calc_user:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalc.g_user = {self.inputs.global_calc_user};\n"
        elif self.inputs.global_calc_mean:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalc.g_mean = {self.inputs.global_calc_mean};\n"
        if self.inputs.no_grand_mean_scaling:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalm.gmsca.gmsca_no = {self.inputs.no_grand_mean_scaling};\n"
        elif self.inputs.grand_mean_scaling:
            snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalm.gmsca.gmsca_yes.gmscv = {self.inputs.grand_mean_scaling};\n"
        snpm_command += f"matlabbatch{{1}}.spm.tools.snpm.des.OneSampT.globalm.glonorm = {self.inputs.global_normalization};\n"

        snpm_command += "matlabbatch{2}.spm.tools.snpm.cp.snpmcfg(1) = cfg_dep('MultiSub: One Sample T test on diffs/contrasts: SnPMcfg.mat configuration file', substruct('.','val', '{}',{1}, '.','val', '{}',{1}, '.','val', '{}',{1}, '.','val', '{}',{1}, '.','val', '{}',{1}), substruct('.','SnPMcfg'));\n"
        snpm_command += "spm('defaults', 'fMRI');\n"
        snpm_command += "spm_jobman('run', matlabbatch);\n"
        snpm_command += "clear matlabbatch\n"

        # Write the SnPM command to a temporary script
        script_path = "/mnt/d/multlearn-sns/SPM/nipype/SnPM_script.m"
        with open(script_path, "w") as script_file:
            script_file.write(snpm_command)

        # Run the SnPM command using subprocess
        subprocess.run(
            f"matlab -nodisplay -nosplash -r \"run('{script_path}'); exit;\"",
            shell=True,
            check=True,
        )

        return runtime

    def _list_outputs(self):
        return self._results

In [51]:
import os
subject_list = range(1,64)
subject_list = [f'{sub:02d}' for sub in subject_list if sub not in [8, 13, 16, 31, 32, 44]]
for contrast in range(1,26):
    SnPM_2nd = Node(SnpmOneSampleTTest(), name='SnPM')
    sub_contrasts = list()
    for sub in subject_list:
        path_1stlevel = f'/mnt/d/multlearn-sns/SPM/nipype/model1/1stLevel/sub-{sub}/con_{contrast:04d}.nii'
        sub_contrasts.append(path_1stlevel)
    SnPM_2nd.inputs.contrasts = sub_contrasts
    destination = '/mnt/d/multlearn-sns/SPM/nipype/model1/2ndLevel/SnPM_SecondLevel_con' + str(contrast)
    if not os.path.exists(destination):
        os.makedirs(destination)
    SnPM_2nd.inputs.destination = destination
    SnPM_2nd.inputs.n_perms = 5000
    SnPM_2nd.inputs.var_smoothing = [0, 0, 0]
    SnPM_2nd.inputs.memory_usage = False
    SnPM_2nd.inputs.cluster_inference_later = -1
    SnPM_2nd.inputs.masking_none = 1
    SnPM_2nd.inputs.implicit_mask = 1
    SnPM_2nd.inputs.global_calc_omit = 1
    SnPM_2nd.inputs.no_grand_mean_scaling = 1
    SnPM_2nd.inputs.global_normalization = 1

    SnPM_2nd.run()

240123-22:13:06,802 nipype.workflow INFO:
	 [Node] Setting-up "SnPM" in "/tmp/tmpe0vzgl0h/SnPM".
240123-22:13:09,40 nipype.workflow INFO:
	 [Node] Executing "SnPM" <__main__.SnpmOneSampleTTest>
240123-22:13:32,25 nipype.workflow INFO:
	 [Node] Finished "SnPM", elapsed time 22.743953s.
240123-22:13:32,241 nipype.workflow WARNING:
	 Storing result file without outputs
240123-22:13:32,607 nipype.workflow WARNING:
	 [Node] Error on "SnPM" (/tmp/tmpe0vzgl0h/SnPM)


NodeExecutionError: Exception raised while executing Node SnPM.

Traceback:
	Traceback (most recent call last):
	  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/interfaces/base/core.py", line 397, in run
	    runtime = self._run_interface(runtime)
	              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
	  File "/tmp/ipykernel_49522/1104522985.py", line 190, in _run_interface
	    subprocess.run(
	  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/subprocess.py", line 550, in run
	    stdout, stderr = process.communicate(input, timeout=timeout)
	                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
	  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/subprocess.py", line 1201, in communicate
	    self.wait()
	  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/subprocess.py", line 1264, in wait
	    return self._wait(timeout=timeout)
	           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
	  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/subprocess.py", l

In [204]:
# SpecifyModel - Generates SPM-specific Model
modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs",
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec",
)

# Level1Design - Generates an SPM design matrix
level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
    ),
    name="level1design",
)

# EstimateModel - estimate the parameters of the model
level1estimate = Node(
    EstimateModel(estimation_method={"Classical": 1}), name="level1estimate"
)

# EstimateContrast - estimates contrasts
level1conest = Node(EstimateContrast(), name="level1conest")

stty: 'standard input': Inappropriate ioctl for device


In [205]:
# Create a Nipype workflow
first_level_wf = Workflow(name='first_level_wf')

# Connect the nodes
first_level_wf.connect([
    (getsubjectinfo, getcontrasts, [('subject_info', 'subject_info')]),
    (getsubjectinfo, modelspec, [('subject_info', 'subject_info')]),
    (getsubjectinfo, modelspec, [('functional_runs', 'functional_runs')]),
    (modelspec, level1design, [('session_info', 'session_info')]),
    (level1design, level1estimate, [('spm_mat_file', 'spm_mat_file')]),  # Connect level1design to level1estimate
    (getcontrasts, level1conest, [('contrasts', 'contrasts')]),  # Connect the contrasts to EstimateContrast
    (level1design, level1conest, [('spm_mat_file', 'spm_mat_file')])
])

getsubjectinfo.inputs.subject = '01'
first_level_wf.config['logging'] = {'workflow_level' : 'DEBUG',
                        'filemanip_level' : 'DEBUG',
                        'interface_level' : 'DEBUG',
                        'log_to_file' : 'True',
                        'log_directory' : '/output/log_folder'}


In [61]:
EstimateContrast?

Init signature: EstimateContrast(**inputs)
Docstring:     
Use spm_contrasts to estimate contrasts of interest

Examples
--------
>>> import nipype.interfaces.spm as spm
>>> est = spm.EstimateContrast()
>>> est.inputs.spm_mat_file = 'SPM.mat'
>>> cont1 = ('Task>Baseline','T', ['Task-Odd','Task-Even'],[0.5,0.5])
>>> cont2 = ('Task-Odd>Task-Even','T', ['Task-Odd','Task-Even'],[1,-1])
>>> contrasts = [cont1,cont2]
>>> est.inputs.contrasts = contrasts
>>> est.run() # doctest: +SKIP
Init docstring: Subclasses must implement __init__
File:           ~/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/interfaces/spm/model.py
Type:           type
Subclasses:     

In [206]:
first_level_wf.run()

240115-17:51:03,760 nipype.workflow INFO:
	 Workflow first_level_wf settings: ['check', 'execution', 'logging', 'monitoring']


240115-17:51:03,938 nipype.workflow INFO:
	 Running serially.
240115-17:51:03,940 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getsubjectinfo" in "/tmp/tmpizpqkkoz/getsubjectinfo".
240115-17:51:03,942 nipype.workflow INFO:
	 [Node] Outdated cache found for "first_level_wf.getsubjectinfo".
240115-17:51:03,981 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240115-17:51:05,42 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 1.05746s.
240115-17:51:05,151 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getcontrasts" in "/tmp/tmpeh8faeip/getcontrasts".
240115-17:51:05,162 nipype.workflow INFO:
	 [Node] Cached "first_level_wf.getcontrasts" - collecting precomputed outputs
240115-17:51:05,163 nipype.workflow INFO:
	 [Node] "first_level_wf.getcontrasts" found cached.
240115-17:51:05,166 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.modelspec" in "/tmp/tmp_jp43wu5/first_level_wf/mode

stty: 'standard input': Inappropriate ioctl for device


240115-17:52:32,711 nipype.workflow INFO:
	 [Node] Finished "level1design", elapsed time 78.98839100000001s.
240115-17:52:32,825 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1estimate" in "/tmp/tmpczr66bpi/first_level_wf/level1estimate".
240115-17:52:32,842 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device


240115-17:55:50,521 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 197.673414s.
240115-17:55:50,531 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1conest" in "/tmp/tmp94o8uqhe/first_level_wf/level1conest".
240115-17:55:50,656 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>
240115-17:55:50,658 nipype.workflow WARNING:
	 [Node] Error on "first_level_wf.level1conest" (/tmp/tmp94o8uqhe/first_level_wf/level1conest)
240115-17:55:50,661 nipype.workflow ERROR:
	 Node level1conest failed to run on host LAPTOP-N61T0IVN.
240115-17:55:50,663 nipype.workflow ERROR:
	 Saving crash info to /mnt/d/multlearn-sns/SPM/code/crash-20240115-175550-ella-level1conest-1cbc120a-b91a-4484-8262-3071f9fedf42.pklz
Traceback (most recent call last):
  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/pipeline/plugins/linear.py", line 47, in run
    node.run(updatehash=updatehash)
  File "/home/ell

ValueError: EstimateContrast requires a value for input 'beta_images'. For a list of required inputs, see EstimateContrast.help()

In [11]:
from nipype.utils.filemanip import loadpkl
res = loadpkl("/mnt/d/multlearn-sns/SPM/nipype/workingdir/first_level_wf/_subject_id_1/getsubjectinfo/result_getsubjectinfo.pklz")

OSError: Result file D:\multlearn-sns\SPM
ipype\workingdirirst_level_wf\_subject_id_1\getsubjectinfoesult_getsubjectinfo.pklz expected, but does not exist after (5.0) seconds.

In [10]:
res

{'node': first_level_wf.getsubjectinfo.a04,
 'traceback': ['Traceback (most recent call last):\n',
  '  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/pipeline/plugins/multiproc.py", line 67, in run_node\n    result["result"] = node.run(updatehash=updatehash)\n                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n',
  '  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/pipeline/engine/nodes.py", line 527, in run\n    result = self._run_interface(execute=True)\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n',
  '  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/pipeline/engine/nodes.py", line 645, in _run_interface\n    return self._run_command(execute)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^\n',
  '  File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/pipeline/engine/nodes.py", line 771, in _run_command\n    raise NodeExecutionError(msg)\n',
  'nipype.pipeline.engine.nodes.NodeExe

In [33]:
File "/home/ella/anaconda3/envs/nipype/lib/python3.11/site-packages/nipype/interfaces/spm/base.py", line 515, in _generate_job
for field in val.dtype.fields:
    TypeError: 'NoneType' object is not iterable


SyntaxError: invalid syntax (1200354413.py, line 1)